In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### XGBoost + Optuna + MLflow Pipeline

In [2]:
#Load datasets

train_hour_df = pd.read_csv("D:/VS Code Projects/Datasets/Bike Sharing/data_processed/feature_engineered_train_hour.csv")
valid_hour_df = pd.read_csv("D:/VS Code Projects/Datasets/Bike Sharing/data_processed/feature_engineered_valid_hour.csv")

In [3]:
### Ensure datetime & remove the column 'Unnamed: 0'
datasets_list =  [train_hour_df,valid_hour_df]
for dataset in datasets_list:
    dataset['date'] = pd.to_datetime(dataset['date'], errors= 'coerce')
    dataset.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

print('ISSUE FIXED')


ISSUE FIXED


In [4]:
target = 'num_rentals'

X_hour_train = train_hour_df.drop(columns= [target, 'date'])
y_hour_train = train_hour_df[target]

X_hour_valid = valid_hour_df.drop(columns= [target, 'date'])
y_hour_valid =valid_hour_df[target]

#### Optimise Hyperparameters with Optuna

In [5]:
def objective_hour(trial):
    params = {
        # Core structure
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.3, log=True),
        
        # Overfitting control
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        
        # Randomness
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
        
        # Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        
        # Fixed parameters
        "random_state": 19,
        "n_jobs": -1,
        "tree_method": "hist"
    }

    with mlflow.start_run(run_name="trial_hour"):
        model = XGBRegressor(**params)
        model.fit(X_hour_train, y_hour_train)
        
        y_pred = model.predict(X_hour_valid)
        rmse = np.sqrt(mean_squared_error(y_hour_valid, y_pred))
        r2 = r2_score(y_hour_valid, y_pred)
        
        # Log hyperparameters and metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "r2": r2})
    
    return rmse

In [6]:
## 1️ Setup MLflow
mlflow.set_tracking_uri(r"file:D:\VS Code Projects\Bike-rental-count-prediction\mlruns")

In [7]:
mlflow.set_experiment("xgboost_optuna_hour")
study_hour = optuna.create_study(
    study_name="xgb_hour_optimisation",
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study_hour.optimize(objective_hour, n_trials=50)

print("Best RMSE (hour):", study_hour.best_value)
print("Best params (hour):", study_hour.best_trial.params)

c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
[I 2026-02-14 11:25:25,745] A new study created in memory with name: xgb_hour_optimisation
[I 2026-02-14 11:25:26,133] Trial 0 finished with value: 42.915246519029374 and parameters: {'n_estimators': 140, 'max_depth': 5, 'learning_rate': 0.08423591097129651, 'min_child_weight': 2, 'gamma': 1.207152897500321, 'subsample': 0.5803000841633539, 'colsample_bytree': 0.8035124590679649, 'reg_alpha': 0.0008161938950154087, 'reg_lambda': 3

Best RMSE (hour): 34.181374065531834
Best params (hour): {'n_estimators': 274, 'max_depth': 3, 'learning_rate': 0.09259116775482727, 'min_child_weight': 1, 'gamma': 0.5341790558293428, 'subsample': 0.5348034992006176, 'colsample_bytree': 0.8832852855505656, 'reg_alpha': 0.00044893875800652546, 'reg_lambda': 7.5469223810468234e-06}


#### Train Final Models with Best Parameters

In [8]:
best_params_hour = study_hour.best_trial.params
model_hour = XGBRegressor(**best_params_hour)
model_hour.fit(X_hour_train, y_hour_train)

y_hour_pred = model_hour.predict(X_hour_valid)
rmse_hour = np.sqrt(mean_squared_error(y_hour_valid, y_hour_pred))
r2_hour = r2_score(y_hour_valid, y_hour_pred)

print("Final Hour Model Results:")
print("RMSE:", rmse_hour)
print("R2:", r2_hour)

with mlflow.start_run(run_name="final_hour_model"):
    mlflow.log_params(best_params_hour)
    mlflow.log_metrics({"rmse": rmse_hour, "r2": r2_hour})
    mlflow.sklearn.log_model(model_hour, "xgb_hour_model")

2026/02/14 11:25:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final Hour Model Results:
RMSE: 34.669677994836135
R2: 0.973004162311554
